In [1]:
import pandas as pd
import numpy as np

DATA_DIR = '/BRISES/data/'
START_DATE = "1981-01-01"
END_DATE = "2021-12-31"

In [2]:
def select_sea_breezes_verbose(data_dir=DATA_DIR):
    """Applies filters to identify Sea Breezes (SBs) days and prints results after each filter."""

    # 1. Initialization
    station_meta = pd.read_csv(data_dir + 'metadata/ERA5_Points_MED.csv', sep=';', encoding='latin1')
    station_ids = list(station_meta['ID'])
    dates = pd.date_range(start=START_DATE, end=END_DATE, freq='D')

    sb_mask = pd.DataFrame(1, columns=station_ids, index=dates)
    sb_mask.index.name = 'dates'

    # -------------------------------------------------------
    # Filter 1: Discard Cyclonic Regimes
    class_JC = pd.read_csv(data_dir + 'filters/F1_JC_classification.csv',index_col='time',parse_dates=True)
    class_JC.index.name = 'dates'
    for i in ['CW', 'CNW', 'CE', 'CNE', 'CN', 'CS', 'CSE', 'CSW']:
        class_JC[class_JC == i] = np.nan
    sb_mask.where(class_JC.notna(), other=np.nan, inplace=True)

#     print("After Filter 1 (Cyclonic Regimes):")
#     print(sb_mask.notna().sum())

    # -------------------------------------------------------
    # Filter 2: Weak geostrophic wind
    geowind = pd.read_csv(data_dir + 'filters/F2_geowind.csv', index_col='dates', parse_dates=True)
    sb_mask.where(geowind <= 12, other=np.nan, inplace=True)

#     print("After Filter 2 (Geostrophic wind <=12 m/s):")
#     print(sb_mask.notna().sum())

    # -------------------------------------------------------
    # Filter 3: Wind direction from the sea
    winddirs = pd.read_csv(data_dir + 'filters/F3_wind_direction.csv', index_col='dates', parse_dates=True)
    winddirs.index.name = 'dates'
    coastal_regions = [['6155A', '6325O'], ['7031', '8025', '8019'], ['8416', '8414A', '8500A', '9981A'], 
                       ['0016A', '76', '0200E'], ['77470'], ['76430', '76500'], ['161530', '76900'],
                       ['161680', '162420', '162450', '162890', '163100'], ['604900'], ['603900', '604020', '603600', '607150'], ['607150']]
    winddirs_regions = [[75, 255], [43, 223], [13, 193], [49, 229], [8, 188], [70, 250], [53, 233], [133, 313],
                        [72, 252], [87, 267], [152, 332]]

    delta = 15

    for i in range(len(coastal_regions)):
        aux = winddirs.loc[:, coastal_regions[i]]

        if winddirs_regions[i][0] < delta:
            aux2 = aux[~(aux[:] >= (winddirs_regions[i][1] + delta) % 360) &
                       (aux[:] <= (winddirs_regions[i][0] - delta) % 360)]
            aux2.index = pd.DatetimeIndex(aux2.index).normalize()
            aux4 = aux2.resample('D').mean()
            for j in aux4.columns:
                aux3 = aux4[j][aux4[j].isna()]
                sb_mask.loc[:, j][sb_mask.index.isin(aux3.index)] = np.nan

        elif winddirs_regions[i][1] > 330:
            aux2 = aux[~(aux[:] >= (winddirs_regions[i][1] - delta) % 360) &
                       (aux[:] <= (winddirs_regions[i][0] + delta) % 360)]
            aux2.index = pd.DatetimeIndex(aux2.index).normalize()
            aux4 = aux2.resample('D').mean()
            for j in aux4.columns:
                aux3 = aux4[j][aux4[j].isna()]
                sb_mask.loc[:, j][sb_mask.index.isin(aux3.index)] = np.nan

        else:
            aux2 = aux[(aux[:] >= (winddirs_regions[i][0] - delta) % 360) &
                       (aux[:] <= (winddirs_regions[i][1] + delta) % 360)]
            aux2.index = pd.DatetimeIndex(aux2.index).normalize()
            aux4 = aux2.resample('D').mean()
            for j in aux4.columns:
                aux3 = aux4[j][aux4[j].isna()]
                sb_mask.loc[:, j][sb_mask.index.isin(aux3.index)] = np.nan

    
#     print("After Filter 3 (Wind direction):")
#     print(sb_mask.notna().sum())

    # -------------------------------------------------------
    # Filter 4: Low cloud cover
    df_clouds = pd.read_csv(data_dir + 'filters/F4_cloud_cover.csv', index_col='dates', parse_dates=True)
    sb_mask.where(df_clouds < 56.25, other=np.nan, inplace=True)

#     print("After Filter 4 (Cloud cover <56.25%):")
#     print(sb_mask.notna().sum())

    # -------------------------------------------------------
    # Filter 5: Low precipitation
    df_p = pd.read_csv(data_dir +'filters/F5_precipitation.csv', index_col='dates', parse_dates=True)
    df_p = df_p.resample("D").sum()
    df_p = df_p.loc[sb_mask.index, sb_mask.columns]
    sb_mask.where(df_p < 0.1, other=np.nan, inplace=True)

#     print("After Filter 5 (Precipitation <0.1 mm):")
#     print(sb_mask.notna().sum())

    return sb_mask


In [3]:
sbs_days = select_sea_breezes_verbose(DATA_DIR)
sbs_days.rename(columns = {'8416':'8416Y'}, inplace=True)
# Save data
# sbs_days.to_csv(DATA_DIR + 'filters/SBs_days_database.csv', sep=',', header=True, index=True, mode='w')

In [4]:
# Count the number of sea breeze days per station
sb_days_count = sbs_days.notna().sum()

# sb_days_count es un Series con estaciones como índice y número de días como valores
print(sb_days_count)

77470     1497
76430     2422
76500     2373
76900     2750
161530    2155
161680    3261
162420    3789
162450    4039
162890    3288
163100    3049
164290    5001
164050    4551
77650     2867
77800     3853
77610     2751
165600    4410
165460    4391
165200    4417
604900    1297
603900     868
604020     830
603600     954
607150    1930
6155A     4708
6325O     5947
7031      5761
8019      4506
8025      5308
8414A     4641
8416Y     4577
8500A     4311
9981A     2402
0016A     2777
76        3215
0200E     3327
B893      4803
B278      4681
B228      4681
B954      5816
dtype: int64
